###  Image processing

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install --upgrade --no-deps tensorflow==2.17.0 opencv-python==4.9.0.80 numpy==1.26.4 pydicom nibabel dicompyler-core matplotlib scikit-learn seaborn tqdm

In [ ]:
!pip install -q pydicom==2.3.1 rt-utils SimpleITK


In [ ]:
import numpy as np
from collections import defaultdict
import pandas as pd
import tensorflow as tf
import cv2
import os
import pydicom
import SimpleITK as sitk
from rt_utils import RTStructBuilder
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import nibabel as nib
from ipywidgets import interact, IntSlider
from tqdm import tqdm


print("Pydicom:", pydicom.__version__)
print("SimpleITK:", sitk.__version__)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)

**Organizing image folders**

```
NSCLC/NSCLC-Radiomics/
    LUNG1-001/
        09-18-2008-StudyID-NA-69331/
            0.000000-NA-82046/ <-- Reconstructed CT with lung filter (high resolution, thin slices)
            3.000000-NA-78236/ <-- Contains patient contours or ROI, usually drawn in radiotherapy software. It is not a series of images, but structures superimposed on the CT.
            300.000000-Segmentation-9.554/ <-- Segmentation mask (SEG) - Defines the tumor region
```

Each patient has a LUNG1-XXX file.

Within it, each study has a folder with the study date.

DICOM: Image data

Computed Tomography (CT): A set of axial slices representing the patient's anatomy. Each CT slice is 3 mm thick.

Segmentation (SEG): Masks that define regions of interest.
Radiotherapy Structures (RTSTRUCT): Outlines drawn by physicians for radiotherapy planning. Associated with a specific CT series.

In [ ]:
# Check the image data types

base_dir = "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics/LUNG1-027/01-01-2014-StudyID-NA-35913"

for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if os.path.isdir(folder_path):
        first_file = os.listdir(folder_path)[0]
        dcm_path = os.path.join(folder_path, first_file)
        ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
        print(folder, "→", ds.Modality)
        print("SeriesDescription:", ds.get("SeriesDescription", "N/A"))
        print("ProtocolName:", ds.get("ProtocolName", "N/A"))
        print("SeriesNumber:", ds.get("SeriesNumber", "N/A"))
        print("Contrast/Bolus Agent:", ds.get("ContrastBolusAgent", "N/A"))
        print("Contrast/Bolus Administration Route:", ds.get("ContrastBolusRoute", "N/A"))
